## Configure Eventstream Routing — Eventhouse + Raw Lakehouse

This notebook walks through routing real-time data from **medical_data_stream**
(Eventstream) to two simultaneous destinations:

```
stream_simulator.py
        │
        ▼
┌───────────────────────────────┐
│  medical_data_stream          │
│  (Eventstream custom endpoint)│
└───────────┬───────────────────┘
            │
     ┌──────┴──────┐
     ▼             ▼
┌──────────┐  ┌────────────────────┐
│Eventhouse│  │ Lakehouse (raw)    │
│medical_  │  │ Hospital_Data_     │
│data_rt_  │  │ Bronze             │
│store     │  │ Tables/streaming/  │
└──────────┘  └────────────────────┘
  KQL tables      Delta tables
```

---

### Workspace Items

| Item | Name |
|------|------|
| Workspace | **CentralData** |
| Eventstream | **medical_data_stream** |
| Eventhouse / KQL DB | **medical_data_rt_store** |
| Raw Lakehouse | **Hospital_Data_Bronze** |

### Step 1 — Add the Custom Endpoint Source (if not yet done)

1. Open **medical_data_stream** in the Fabric portal
2. In the canvas, click **New source → Custom endpoint**
3. Name it `nursing_realtime_source`
4. After creation, click the source node and copy:
   - **Event Hub compatible connection string** → paste into `simulator/.env` as `EVENT_HUB_CONNECTION_STRING`
   - **Event Hub name** → paste into `simulator/.env` as `EVENT_HUB_NAME`

---

### Step 2 — Add Eventhouse Destination

1. From the Eventstream canvas, click **New destination → Eventhouse**
2. Configure:
   - **Eventhouse**: `medical_data_rt_store`
   - **KQL Database**: `medical_data_rt_store` (default DB)
   - **Destination table**: `streaming_events` (or create individual tables per stream type)
3. In **Data format**, select **JSON**
4. Under **Column mapping**, the Eventstream will auto-detect columns from incoming data.
   For a single combined table, ensure these columns are mapped:
   - `stream_type` (string) — identifies the source: `rtls_location`, `ehr_clickstream`, etc.
   - `data` (dynamic/JSON) — the full event payload

   **Alternative: Per-stream tables** — Use the Eventstream **Group by** or **Filter** transform
   to route each `stream_type` to a dedicated KQL table:

   | stream_type | KQL Table |
   |---|---|
   | `rtls_location` | `stream_rtls_location` |
   | `ehr_clickstream` | `stream_ehr_clickstream` |
   | `nurse_call_events` | `stream_nurse_call_events` |
   | `bcma_scans` | `stream_bcma_scans` |
   | `clinical_alerts` | `stream_clinical_alerts` |

5. Click **Add** → **Activate**

---

### Step 3 — Add Lakehouse Destination (simultaneous)

1. From the same Eventstream canvas, click **New destination → Lakehouse**
2. Configure:
   - **Lakehouse**: `Hospital_Data_Bronze`
   - **Delta table name**: `raw_streaming_events`
   - **Input data format**: JSON
3. Under **Column mapping**, map all incoming fields
4. Click **Add** → **Activate**

Both destinations run in parallel — every event entering the Eventstream
is delivered to **both** the Eventhouse and the Lakehouse simultaneously.

---

### Step 4 — Run the Simulator

```bash
cd datasets/healthcare_nursing_documentation/simulator
pip install -r requirements.txt
cp .env.example .env  # edit with your connection string + event hub name
python stream_simulator.py
```

In [ ]:
# ── Verify: Eventhouse — query the KQL database for recent streaming data ──
#
# Run this cell from a notebook attached to the medical_data_rt_store Eventhouse,
# or paste the queries directly into the KQL queryset in Fabric.

kql_verify = """
// ── 1. Row counts per stream type (combined table approach)
streaming_events
| summarize EventCount = count() by stream_type
| order by EventCount desc

// ── 2. Events received in the last 30 minutes
streaming_events
| where ingestion_time() > ago(30m)
| summarize Count = count() by bin(ingestion_time(), 1m), stream_type
| render timechart

// ── 3. Per-table approach — if you created separate KQL tables
// stream_rtls_location      | count
// stream_ehr_clickstream     | count
// stream_nurse_call_events   | count
// stream_bcma_scans          | count
// stream_clinical_alerts     | count
"""
print(kql_verify)

In [ ]:
# ── Verify: Lakehouse — confirm raw streaming data arrived in Delta table ──
#
# Run this cell from a notebook attached to Hospital_Data_Bronze.

df_raw = spark.read.format("delta").load("Tables/raw_streaming_events")

print(f"Total rows in raw_streaming_events: {df_raw.count()}")
print(f"Schema:")
df_raw.printSchema()

# Breakdown by stream type
df_raw.groupBy("stream_type").count().orderBy("count", ascending=False).show()

# Show latest 5 events
df_raw.orderBy(df_raw["_timestamp"].desc()).limit(5).show(truncate=False)

### Architecture Summary

| Component | Purpose | Data |
|-----------|---------|------|
| **stream_simulator.py** | Replays 5 streaming CSVs as JSON events via Event Hub protocol | 360 events (rtls: 95, ehr: 135, calls: 35, bcma: 57, alerts: 38) |
| **medical_data_stream** | Fabric Eventstream — ingests + routes in real time | All 5 stream types |
| **medical_data_rt_store** | Eventhouse — KQL analytics, sub-second queries | `streaming_events` table (or 5 per-type tables) |
| **Hospital_Data_Bronze** | Lakehouse — raw Delta archive for Spark/SQL analytics | `raw_streaming_events` table |

#### Monitoring Tips
- Open the Eventstream **Metrics** tab to see throughput, latency, and error rates per destination
- Use the **Data preview** in each destination node to confirm events are arriving
- For alerting, add a **Reflex** destination to trigger actions on specific event patterns (e.g., critical clinical alerts)